# Smart Grid Load Forecasting - Model Training

This notebook contains the clean final training workflow for the smart grid load forecasting model.

The final model uses safe forecasting features such as time features, lag features, rolling averages, weather, renewable energy and electricity price variables.

Same-timestamp electrical proxy variables such as Current (A), Voltage (V), Reactive Power, Power Factor and Grid Supply are excluded to reduce potential leakage.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Paths

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "smart_grid_dataset.csv"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(exist_ok = True)

MODEL_PATH = MODELS_DIR / "load_forecasting_model.joblib"
CONFIG_PATH = MODELS_DIR / "model_config.json"

DATA_PATH, MODEL_PATH , CONFIG_PATH


(WindowsPath('c:/Users/zaaaa/OneDrive/Υπολογιστής/New folder/Python Projects/Smart Grid Load Forecasting & Anomaly Detection API/data/smart_grid_dataset.csv'),
 WindowsPath('c:/Users/zaaaa/OneDrive/Υπολογιστής/New folder/Python Projects/Smart Grid Load Forecasting & Anomaly Detection API/models/load_forecasting_model.joblib'),
 WindowsPath('c:/Users/zaaaa/OneDrive/Υπολογιστής/New folder/Python Projects/Smart Grid Load Forecasting & Anomaly Detection API/models/model_config.json'))

## Constants

In [3]:
TIMESTAMP_COL = "Timestamp"
TARGET = "Power Consumption (kW)"

SAFE_FEATURES = [
    "Solar Power (kW)",
    "Wind Power (kW)",
    "Temperature (°C)",
    "Humidity (%)",
    "Electricity Price (USD/kWh)",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_4",
    "lag_96",
    "rolling_mean_4",
    "rolling_mean_24",
    "rolling_mean_96",
]

## Load dataset

In [4]:
df = pd.read_csv(DATA_PATH)

df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

df.shape

(50000, 16)

## Feature engineering function

In [ ]:
def create_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    df_model = dataframe.copy()

    # Time-based features
    df_model["hour"] = df_model[TIMESTAMP_COL].dt.hour
    df_model["day_of_week"] = df_model[TIMESTAMP_COL].dt.dayofweek
    df_model["month"] = df_model[TIMESTAMP_COL].dt.month
    df_model["is_weekend"] = df_model["day_of_week"].isin([5, 6]).astype(int)

    # Lag features
    df_model["lag_1"] = df_model[TARGET].shift(1)
    df_model["lag_4"] = df_model[TARGET].shift(4)
    df_model["lag_96"] = df_model[TARGET].shift(96)

    # Rolling mean features
    df_model["rolling_mean_4"] = df_model[TARGET].shift(1).rolling(window=4).mean()
    df_model["rolling_mean_24"] = df_model[TARGET].shift(1).rolling(window=24).mean()
    df_model["rolling_mean_96"] = df_model[TARGET].shift(1).rolling(window=96).mean()

    # Remove rows created with NaN values due to lag/rolling features
    df_model = df_model.dropna().reset_index(drop=True)

    return df_model

## Create model dataframe

In [6]:
df_model = create_features(df)

df_model.shape

(49809, 26)

## Prepare X and y

In [7]:
X = df_model[SAFE_FEATURES]
y = df_model[TARGET]

X.shape, y.shape

((49809, 15), (49809,))

## Chronological train/test split

In [8]:
split_index = int(len(df_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((39847, 15), (9962, 15), (39847,), (9962,))

## Naive baseline

In [9]:
baseline_predictions = X_test["lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

print(f"Naive Baseline MAE: {baseline_mae:.4f}")
print(f"Naive Baseline RMSE: {baseline_rmse:.4f}")



Naive Baseline MAE: 3.4734
Naive Baseline RMSE: 4.2515


## Train final model

In [10]:
model = HistGradientBoostingRegressor(max_iter=200, random_state=42)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

model_mae = mean_absolute_error(y_test, predictions)
model_rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Final Model MAE: {model_mae:.4f}")
print(f"Final Model RMSE: {model_rmse:.4f}")



Final Model MAE: 2.5949
Final Model RMSE: 2.9977


## Compare baseline vs final model

In [11]:
mae_improvement = ((baseline_mae - model_mae) / baseline_mae) * 100
rmse_improvement = ((baseline_rmse - model_rmse) / baseline_rmse) * 100

comparison_df = pd.DataFrame([
    {"model": "Naive Baseline(lag_1)",
    "mae": baseline_mae,
    "rmse": baseline_rmse},

    {"model": "HistGradientBoostingRegressor",
     "mae": model_mae,
     "rmse": model_rmse}
])

print(f"MAE improvement over baseline: {mae_improvement:.2f}%")
print(f"RMSE improvement over baseline: {rmse_improvement:.2f}%")

MAE improvement over baseline: 25.29%
RMSE improvement over baseline: 29.49%


## Anomaly threshold from training residuals

In [12]:
train_predictions = model.predict(X_train)

train_residuals = y_train.values - train_predictions
train_abs_residuals = np.abs(train_residuals)

anomaly_threshold = float(np.percentile(train_abs_residuals, 95))

print(f"Anomaly threshold: {anomaly_threshold:.4f}")


Anomaly threshold: 4.8887


## Test anomaly summary

In [13]:
test_results = df_model.iloc[split_index:].copy()

test_results["actual_power"] = y_test.values
test_results["predicted_power"] = predictions
test_results["residual"] = test_results["actual_power"] - test_results["predicted_power"]
test_results["abs_residual"] = test_results["residual"].abs()

test_results["is_anomaly"] = (test_results["abs_residual"] > anomaly_threshold).astype(int)

anomaly_rate = test_results["is_anomaly"].mean() * 100

print(test_results["is_anomaly"].value_counts())
print(f"Anomaly rate in test set: {anomaly_rate:.2f}%")


is_anomaly
0    9393
1     569
Name: count, dtype: int64
Anomaly rate in test set: 5.71%


## Save model and config

In [14]:
joblib.dump(model, MODEL_PATH)

model_config = { "model_type": "HistGradientBoostingRegressor",
    "target_column": TARGET,
    "timestamp_column": TIMESTAMP_COL,
    "feature_columns": SAFE_FEATURES,
    "train_test_split": "chronological_80_20",
    "anomaly_threshold": float(anomaly_threshold),
    "metrics": {
        "baseline_mae": float(baseline_mae),
        "baseline_rmse": float(baseline_rmse),
        "model_mae": float(model_mae),
        "model_rmse": float(model_rmse),
        "mae_improvement_percent": float(mae_improvement),
        "rmse_improvement_percent": float(rmse_improvement),
        "test_anomaly_rate_percent": float(anomaly_rate)
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=4)

MODEL_PATH, CONFIG_PATH

(WindowsPath('c:/Users/zaaaa/OneDrive/Υπολογιστής/New folder/Python Projects/Smart Grid Load Forecasting & Anomaly Detection API/models/load_forecasting_model.joblib'),
 WindowsPath('c:/Users/zaaaa/OneDrive/Υπολογιστής/New folder/Python Projects/Smart Grid Load Forecasting & Anomaly Detection API/models/model_config.json'))

## Verify saved artifacts

In [15]:
loaded_model = joblib.load(MODEL_PATH)

loaded_model = joblib.load(MODEL_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    loaded_config = json.load(file)

loaded_config

{'model_type': 'HistGradientBoostingRegressor',
 'target_column': 'Power Consumption (kW)',
 'timestamp_column': 'Timestamp',
 'feature_columns': ['Solar Power (kW)',
  'Wind Power (kW)',
  'Temperature (°C)',
  'Humidity (%)',
  'Electricity Price (USD/kWh)',
  'hour',
  'day_of_week',
  'month',
  'is_weekend',
  'lag_1',
  'lag_4',
  'lag_96',
  'rolling_mean_4',
  'rolling_mean_24',
  'rolling_mean_96'],
 'train_test_split': 'chronological_80_20',
 'anomaly_threshold': 4.88865720313224,
 'metrics': {'baseline_mae': 3.4734226726543285,
  'baseline_rmse': 4.251530918851246,
  'model_mae': 2.5948536578991668,
  'model_rmse': 2.9977014881618333,
  'mae_improvement_percent': 25.294042722528054,
  'rmse_improvement_percent': 29.491245732918113,
  'test_anomaly_rate_percent': 5.711704477012648}}